# Posiciones de la Tierra y del Sol respecto al centro de la Luna

Consulta directa a [JPL Horizons API](https://ssd-api.jpl.nasa.gov/doc/horizons.html), sin clave API.
Se realizan dos peticiones consecutivas: Tierra (`399`) y Sol (`10`), con el centro de la Luna
como origen (`CENTER=500@301`).

**Convenciones:** posiciones geométricas instantáneas, en kilómetros, sin corrección de tiempo
de viaje de la luz ni aberración (`VEC_CORR=NONE`). Los ejes son los de ICRF (`REF_PLANE=FRAME`):
el origen sigue a la Luna, pero los ejes no giran con su superficie. No son coordenadas de latitud/longitud lunar.
Para las fechas modernas se usa UTC de forma explícita (`TIME_TYPE=UT`), evitando el TDB predeterminado.
Horizons aplica los saltos de segundo conocidos para fechas futuras.

Ejecuta las celdas en orden. Cambia **INICIO_UTC**, **DURACION** y **PASO** en la siguiente celda.
Las dependencias `requests`, `numpy` y `pandas` ya están declaradas en el proyecto.


In [1]:
from datetime import timedelta
import csv
import io

import numpy as np
import pandas as pd
import requests

# Fecha y hora UTC, no hora local de España. También admite un offset explícito.
INICIO_UTC = "2026-11-14 00:00:00"
DURACION = timedelta(days=1)       # Ejemplos: hours=6, days=7, minutes=90
PASO = timedelta(hours=1)          # Ejemplos: minutes=10, hours=2, days=1

# Paso: un número entero de minutos, como mínimo 1 minuto.
# Si el final no coincide con un paso, la última muestra queda antes del final.
# Si el final coincide, se incluye (1 día cada hora -> 25 muestras).


In [2]:
HORIZONS_URL = "https://ssd.jpl.nasa.gov/api/horizons.api"
COLUMNAS_POSICION = [
    "tierra_x_km", "tierra_y_km", "tierra_z_km",
    "sol_x_km", "sol_y_km", "sol_z_km",
]


def posiciones_desde_luna(inicio_utc, duracion, paso):
    """Devuelve una tabla con tiempo UTC, segundos desde el inicio y posiciones ICRF en km.

    Fechas desde 1962 (UT equivale a UTC en Horizons). Límite local de 10000 muestras.
    Las fechas sin zona se interpretan como UTC; las que tienen zona se convierten a UTC.
    """
    inicio = pd.Timestamp(inicio_utc)
    if pd.isna(inicio):
        raise ValueError("El inicio debe ser una fecha válida.")
    inicio = inicio.tz_localize("UTC") if inicio.tzinfo is None else inicio.tz_convert("UTC")
    if inicio.year < 1962:
        raise ValueError("Usa fechas desde 1962 para que UT corresponda a UTC.")
    if inicio.microsecond or inicio.nanosecond:
        raise ValueError("Especifica el inicio con precisión de segundos enteros.")
    duracion = pd.Timedelta(duracion)
    paso = pd.Timedelta(paso)
    if pd.isna(duracion) or duracion <= pd.Timedelta(0):
        raise ValueError("La duración debe ser positiva.")
    if duracion % pd.Timedelta(seconds=1):
        raise ValueError("La duración debe ser un número entero de segundos.")
    if pd.isna(paso) or paso < pd.Timedelta(minutes=1) or paso % pd.Timedelta(minutes=1):
        raise ValueError("El paso debe ser un número entero de minutos, como mínimo 1.")
    if duracion < paso:
        raise ValueError("La duración debe ser al menos igual al paso.")
    n = int(duracion // paso) + 1
    if n > 10000:
        raise ValueError(f"Se solicitan {n} muestras; aumenta el paso o reduce la duración (máximo local: 10000).")
    fin = inicio + duracion
    esperados = pd.date_range(inicio, periods=n, freq=paso)

    parametros = {
        "format": "json",
        "OBJ_DATA": "NO",
        "MAKE_EPHEM": "YES",
        "EPHEM_TYPE": "VECTORS",
        "CENTER": "500@301",
        "START_TIME": inicio.strftime("%Y-%m-%d %H:%M:%S"),
        "STOP_TIME": fin.strftime("%Y-%m-%d %H:%M:%S"),
        "STEP_SIZE": f"{int(paso // pd.Timedelta(minutes=1))} m",
        "TIME_TYPE": "UT",
        "TIME_DIGITS": "FRACSEC",
        "CAL_TYPE": "GREGORIAN",
        "REF_SYSTEM": "ICRF",
        "REF_PLANE": "FRAME",
        "VEC_CORR": "NONE",
        "OUT_UNITS": "KM-S",
        "VEC_TABLE": "1",
        "CSV_FORMAT": "YES",
    }

    def consultar(sesion, cuerpo, identificador):
        # Requests codifica los parámetros; Horizons requiere comillas en sus valores.
        consulta = {k: (v if k == "format" else f"'{v}'") for k, v in parametros.items()}
        consulta["COMMAND"] = f"'{identificador}'"
        try:
            respuesta = sesion.get(HORIZONS_URL, params=consulta, timeout=(15, 120))
            respuesta.raise_for_status()
            contenido = respuesta.json()
        except (requests.RequestException, ValueError) as error:
            raise RuntimeError(f"No se pudo consultar Horizons para {cuerpo}: {error}") from error
        if contenido.get("error"):
            raise RuntimeError(f"Horizons ({cuerpo}): {contenido['error']}")
        texto = contenido.get("result", "")
        if "$$SOE" not in texto or "$$EOE" not in texto:
            raise RuntimeError(f"Horizons no devolvió vectores para {cuerpo}: {texto[:2000]}")
        bloque = texto.split("$$SOE", 1)[1].split("$$EOE", 1)[0].strip()
        filas = [fila for fila in csv.reader(io.StringIO(bloque)) if fila and fila[0].strip()]
        if len(filas) != n or any(len(fila) < 5 for fila in filas):
            raise RuntimeError(f"Tabla incompleta o inesperada para {cuerpo}: {len(filas)} filas, esperadas {n}.")
        fechas = pd.to_datetime(
            [fila[1].strip().removeprefix("A.D. ") for fila in filas],
            format="%Y-%b-%d %H:%M:%S.%f", utc=True,
        )
        if not fechas.equals(esperados):
            raise RuntimeError(f"Las fechas devueltas para {cuerpo} no coinciden con el muestreo solicitado.")
        xyz = np.array([[float(valor) for valor in fila[2:5]] for fila in filas])
        if not np.isfinite(xyz).all():
            raise RuntimeError(f"Horizons devolvió posiciones no finitas para {cuerpo}.")
        return xyz

    # Peticiones secuenciales, siguiendo la política de uso de las APIs de SSD.
    with requests.Session() as sesion:
        tierra = consultar(sesion, "Tierra", 399)
        sol = consultar(sesion, "Sol", 10)

    tabla = pd.DataFrame(np.column_stack((tierra, sol)), columns=COLUMNAS_POSICION)
    tabla.insert(0, "t_segundos", (esperados - inicio).total_seconds())
    tabla.insert(0, "tiempo_utc", esperados)
    tabla.attrs.update(origen="Centro de la Luna (301)", marco="ICRF / FRAME",
                       unidades="km", escala_temporal="UTC", correccion="NONE")
    return tabla


In [3]:
# Esta celda llama realmente a la API. Vuelve a ejecutarla tras cambiar la configuración.
tabla = posiciones_desde_luna(INICIO_UTC, DURACION, PASO)

# Array numérico de forma (N, 7); t_segundos se mide desde INICIO_UTC.
COLUMNAS_ARRAY = ["t_segundos", *COLUMNAS_POSICION]
datos = tabla[COLUMNAS_ARRAY].to_numpy(dtype=float)
tiempos_utc = tabla["tiempo_utc"].to_numpy()

# Arrays separados, si los necesitas para cálculos o animaciones:
posiciones_tierra_km = datos[:, 1:4]   # (N, 3)
posiciones_sol_km = datos[:, 4:7]      # (N, 3)

print(f"Muestras: {len(tabla)} | Forma del array: {datos.shape}")
print(f"Desde {tabla['tiempo_utc'].iloc[0]} hasta {tabla['tiempo_utc'].iloc[-1]}")
print("Columnas:", COLUMNAS_ARRAY)
display(tabla.head())
display(datos[:5])


Muestras: 25 | Forma del array: (25, 7)
Desde 2026-11-14 00:00:00+00:00 hasta 2026-11-15 00:00:00+00:00
Columnas: ['t_segundos', 'tierra_x_km', 'tierra_y_km', 'tierra_z_km', 'sol_x_km', 'sol_y_km', 'sol_z_km']


,tiempo_utc,t_segundos,tierra_x_km,tierra_y_km,tierra_z_km,sol_x_km,sol_y_km,sol_z_km
0,2026-11-14 00:00:00+00:00,0.0,-92440.720151,352267.637064,178495.847048,-9.274648e+07,-1.055756e+08,-4.573875e+07
1,2026-11-14 01:00:00+00:00,3600.0,-95819.351274,351618.499134,177965.199436,-9.266442e+07,-1.056374e+08,-4.576580e+07
2,2026-11-14 02:00:00+00:00,7200.0,-99190.174074,350941.893290,177420.628954,-9.258230e+07,-1.056992e+08,-4.579284e+07
3,2026-11-14 03:00:00+00:00,10800.0,-102552.924453,350237.867308,176862.175933,-9.250012e+07,-1.057609e+08,-4.581986e+07
4,2026-11-14 04:00:00+00:00,14400.0,-105907.338796,349506.470733,176289.881607,-9.241789e+07,-1.058227e+08,-4.584688e+07


array([[ 0.00000000e+00, -9.24407202e+04,  3.52267637e+05,
         1.78495847e+05, -9.27464834e+07, -1.05575551e+08,
        -4.57387531e+07],
       [ 3.60000000e+03, -9.58193513e+04,  3.51618499e+05,
         1.77965199e+05, -9.26644192e+07, -1.05637378e+08,
        -4.57658008e+07],
       [ 7.20000000e+03, -9.91901741e+04,  3.50941893e+05,
         1.77420629e+05, -9.25822982e+07, -1.05699177e+08,
        -4.57928378e+07],
       [ 1.08000000e+04, -1.02552924e+05,  3.50237867e+05,
         1.76862176e+05, -9.25001202e+07, -1.05760946e+08,
        -4.58198642e+07],
       [ 1.44000000e+04, -1.05907339e+05,  3.49506471e+05,
         1.76289882e+05, -9.24178850e+07, -1.05822686e+08,
        -4.58468799e+07]])

## Uso de los resultados

Cada fila de `datos` contiene:

`[t_segundos, tierra_x_km, tierra_y_km, tierra_z_km, sol_x_km, sol_y_km, sol_z_km]`

`tiempos_utc[i]` es la fecha UTC de la fila `i`. `tabla` conserva ambos tiempos junto con
las seis coordenadas. La Luna está siempre en `(0, 0, 0)` en este sistema.
Para obtener distancias al centro lunar: `np.linalg.norm(posiciones_tierra_km, axis=1)`
o `np.linalg.norm(posiciones_sol_km, axis=1)`.

Ejemplos de configuración:

| Duración | Muestreo | Variables |
| --- | --- | --- |
| 6 horas | Cada 10 minutos | `DURACION = timedelta(hours=6)`; `PASO = timedelta(minutes=10)` |
| 7 días | Cada 2 horas | `DURACION = timedelta(days=7)`; `PASO = timedelta(hours=2)` |
| 30 días | Cada día | `DURACION = timedelta(days=30)`; `PASO = timedelta(days=1)` |

Con el mismo inicio, duración y paso, vuelve a ejecutar la celda de consulta para regenerar los resultados.
Si lo necesitas, puedes exportarlos con `tabla.to_csv("posiciones_luna.csv", index=False)`
o `np.save("posiciones_luna.npy", datos)`.

Referencias: [parámetros de la API](https://ssd-api.jpl.nasa.gov/doc/horizons.html)
y [sistemas de referencia y tiempo](https://ssd.jpl.nasa.gov/horizons/manual.html).


In [4]:
tabla

,tiempo_utc,t_segundos,tierra_x_km,tierra_y_km,tierra_z_km,sol_x_km,sol_y_km,sol_z_km
0,2026-11-14 00:00:00+00:00,0.0,-92440.720151,352267.637064,178495.847048,-9.274648e+07,-1.055756e+08,-4.573875e+07
1,2026-11-14 01:00:00+00:00,3600.0,-95819.351274,351618.499134,177965.199436,-9.266442e+07,-1.056374e+08,-4.576580e+07
2,2026-11-14 02:00:00+00:00,7200.0,-99190.174074,350941.893290,177420.628954,-9.258230e+07,-1.056992e+08,-4.579284e+07
3,2026-11-14 03:00:00+00:00,10800.0,-102552.924453,350237.867308,176862.175933,-9.250012e+07,-1.057609e+08,-4.581986e+07
4,2026-11-14 04:00:00+00:00,14400.0,-105907.338796,349506.470733,176289.881607,-9.241789e+07,-1.058227e+08,-4.584688e+07
5,2026-11-14 05:00:00+00:00,18000.0,-109253.153976,348747.754875,175703.788108,-9.233559e+07,-1.058844e+08,-4.587388e+07
6,2026-11-14 06:00:00+00:00,21600.0,-112590.107363,347961.772805,175103.938463,-9.225324e+07,-1.059461e+08,-4.590088e+07
7,2026-11-14 07:00:00+00:00,25200.0,-115917.936836,347148.579355,174490.376594,-9.217083e+07,-1.060077e+08,-4.592786e+07
8,2026-11-14 08:00:00+00:00,28800.0,-119236.380787,346308.231113,173863.147316,-9.208837e+07,-1.060694e+08,-4.595483e+07
9,2026-11-14 09:00:00+00:00,32400.0,-122545.178137,345440.786423,173222.296335,-9.200584e+07,-1.061309e+08,-4.598180e+07
